In [21]:
import pandas as pd
import requests
from bs4 import BeautifulSoup as bs
from tqdm import tqdm
import regex as re
import os
import json

from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS

from get_viaf_label import VIAFClient


BASE_URI = 'http://bibframe.example.org/'

In [13]:
entities_dict = {
        'authors': {},
        'subjects': {},
        'genreforms': {},
    }

flow_control = {
        'work_last_id': 0,
        'instance_last_id': 0,
        'item_last_id': 0,
        'topic_last_id': 0,
        'genreform_last_id': 0,
        'author_last_id': 0,
    }

In [31]:
def manual_df_to_dict(path):
    df = pd.read_excel(path, sheet_name='Posts').fillna('')
    df = df[df['do PBL'] == True]
    return df.to_dict(orient='records')

def clear_viaf_uri(url):
    if uri := re.match(r'https://viaf\.org/viaf/\d+', url):
        return uri.group(0) + '/'
    
def get_geonames_label(url):
    label = url.split('/')[-1].replace('.html', '').capitalize()
    return label

def get_filmpolski_label(url):
    response = requests.get(url)
    if response.ok:
        response.encoding = 'utf-8'
        soup = bs(response.text, 'lxml')
        label = soup.find('article', {'id': 'film'}).find('h1').text
        return label
    
def preprocess_forms():
    forms = [
                'artykuł', 
                'esej',
                'felieton',
                'inne',
                'kalendarium',
                'kult',
                'list',
                'miniatura prozą',
                'nota',
                'opowiadanie',
                'poemat',
                'proza',
                'proza poetycka',
                'recenzja',
                'reportaż',
                'rozmyślanie religijne',
                'scenariusz',
                'słownik',
                'sprostowanie',
                'szkic',
                'teksty dramatyczne',
                'wiersz',
                'wpis blogowy',
                'wspomnienie',
                'wypowiedź',
                'wywiad',
                'zgon',
            ]
    
    for idx, form in enumerate(forms):
        form_uri = BASE_URI + 'genreForms/genreform' + str(idx + 1).zfill(8)
        entities_dict['genreforms'][form] = form_uri

def preprocess_authors(records, with_viaf=False):
    authors_list = set([(rec.get('Autor'), rec.get('VIAF autor 1'), rec.get('VIAF autor 2'), rec.get('VIAF autor 3')) for rec in records])
    with VIAFClient(headless=True) as viaf_client:
        for author_tuple in tqdm(authors_list, desc="Preprocessing authors..."):
            if author_tuple[0]:
                author_splitted = author_tuple[0].split('|')
            else: continue
            
            for idx, aut in enumerate(author_splitted):
                if not aut:
                    continue
                if idx > 2: break
                label = aut.strip()
                if (viaf_url := author_tuple[idx + 1]):
                    if with_viaf:
                        # viaf_label = viaf_client.get_label(viaf_url)
                        pass
                    else: viaf_label = None
                    viaf_uri = clear_viaf_uri(viaf_url)
                else:
                    viaf_label = None
                    viaf_uri = None   
                if label not in entities_dict['authors']:
                    last_id = flow_control['author_last_id']
                    author_id = str(last_id + 1).zfill(8)
                    entities_dict['authors'][label] = {
                            'author_id': author_id,
                            'viaf_uri': viaf_uri,
                            'viaf_label': viaf_label,
                            'bibframe_type': 'bf:Agent',
                        }
                    flow_control['author_last_id'] += 1

def preprocess_authors_automatic(records):
    authors_list = set([rec.get('Autor') for rec in records])
    for author_field in authors_list:
        if author_field:
            for author in author_field.split('|'):
                author = author.strip()
                if author not in entities_dict['authors']:
                    last_id = flow_control['author_last_id']
                    author_id = str(last_id + 1).zfill(8)
                    entities_dict['authors'][author] = {
                            'author_id': author_id,
                            'viaf_uri': '',
                            'viaf_label': '',
                            'bibframe_type': 'bf:Agent',
                        }
                    flow_control['author_last_id'] += 1

def preprocess_topics(records):
    topics_map = {
            'czasopismo': 'Work',
            'film': 'MovingImage',
            'instytucja': 'Organization',
            'kraj': 'Place',
            'książka': 'Work',
            'miejscowość': 'Place',
            'osoba': 'Person',
            'spektakl': 'Work',
            'wydarzenie': 'Event',
        }
    
    # entities
    entities_list = [[(rec.get('byt 1'), rec.get('zewnętrzny identyfikator bytu 1')), 
                     (rec.get('byt 2'), rec.get('zewnętrzny identyfikator bytu 2')), 
                     (rec.get('byt 3'), rec.get('zewnętrzny identyfikator bytu 3'))]
                     for rec in records]
    entities_list = [e for sublist in entities_list for e in sublist]
    entities_list = [e for e in entities_list if e[0] and e[1]]
    
    with VIAFClient(headless=True) as viaf_client:
        for elem in tqdm(entities_list, desc="Preprocessing topics..."):
            if elem[1].startswith('https://viaf.org/'):
                if clear_viaf_uri(elem[1]) in entities_dict['subjects']:
                    continue
                try:
                    label = viaf_client.get_label(elem[1])
                except: continue
                if label:
                    uri = clear_viaf_uri(elem[1])
                    key = uri
                    try:
                        bibframe_type = 'bf:' + topics_map[elem[0]]
                    except: continue
                else:
                    continue
            elif elem[1].startswith('https://filmpolski.pl/'):
                if elem[1] in entities_dict['subjects']:
                    continue
                try:
                    label = get_filmpolski_label(elem[1])
                except: continue
                if label:
                    uri = elem[1]
                    key = uri
                    try:
                        bibframe_type = 'bf:' + topics_map[elem[0]]
                    except: continue
                else:
                    continue
            elif elem[1].startswith('https://www.geonames.org/'):
                if elem[1] in entities_dict['subjects']:
                    continue
                label = get_geonames_label(elem[1])
                if label:
                    uri = elem[1]
                    key = uri
                    try:
                        bibframe_type = 'bf:' + topics_map[elem[0]]
                    except: continue
                else:
                    continue
            else:
                continue

            if key and key not in entities_dict['subjects']:
                last_id = flow_control['topic_last_id']
                topic_uri = BASE_URI + 'subjects/subject' + str(last_id + 1).zfill(8)
                entities_dict['subjects'][key] = {
                        'label': label,
                        'uri': topic_uri,
                        'external_uri': uri,
                        'bibframe_type': bibframe_type,
                    }
                flow_control['topic_last_id'] += 1

def preprocess_tags(records):
    try:
        topics = [rec.get('Tagi').split('|') for rec in records if rec.get('Tagi')]
        topics = list(set([e.strip() for sublist in topics for e in sublist]))
        for topic in topics:
            if topic and topic not in entities_dict['subjects']:
                last_id = flow_control['topic_last_id']
                topic_id = str(last_id + 1).zfill(8)
                entities_dict['subjects'][topic] = {
                        'topic_id': topic_id,
                        'external_uri': None,
                        'bibframe_type': 'bf:Topic',
                    }
                flow_control['topic_last_id'] += 1
    except KeyError:
        pass

In [32]:
with open('entities/processed_files.json', 'r', encoding='utf-8') as jfile:
    processed_files = json.load(jfile)
    processed_files = set(processed_files)

with open('entities/entities_dict.json', 'r', encoding='utf-8') as jfile:
    entities_dict = json.load(jfile)

with open('entities/flow_control.json', 'r', encoding='utf-8') as jfile:
    flow_control = json.load(jfile)

In [ ]:
# entities from manual
list_of_files = os.listdir('to_bibframe/manual/')
for filename in tqdm(list_of_files):
    if filename in processed_files:
        continue
    print(filename)
    path = 'to_bibframe/manual/' + filename
    records = manual_df_to_dict(path)
    preprocess_forms()
    preprocess_authors(records, with_viaf=False)
    preprocess_topics(records)
    preprocess_tags(records)
    with open('entities_dict.json', 'w', encoding='utf-8') as jfile:
        json.dump(entities_dict, jfile, ensure_ascii=False)
    with open('flow_control.json', 'w', encoding='utf-8') as jfile:
        json.dump(flow_control, jfile, ensure_ascii=False)


In [ ]:
# entities from automatic 
list_of_files = os.listdir('to_bibframe/automatic_full/')
for filename in tqdm(list_of_files):
    print(filename)
    path = 'to_bibframe/automatic_full/' + filename
    with open(path, 'r', encoding='utf-8') as json_file:
        records = json.load(json_file)
    preprocess_authors_automatic(records)
    preprocess_tags(records)

with open('entities/entities_dict.json', 'w', encoding='utf-8') as jfile:
    json.dump(entities_dict, jfile, ensure_ascii=False)
with open('entities/flow_control.json', 'w', encoding='utf-8') as jfile:
    json.dump(flow_control, jfile, ensure_ascii=False)